# Volume 2 Drills — Tabular Methods

15 drills covering Temporal Difference learning, SARSA, Q-learning, Monte Carlo methods,  
and the key distinctions between on-policy and off-policy algorithms.

**Prerequisites:** Volume 1 drills, Chapters 4–7 of Sutton & Barto.

---
## Drill 1 — TD(0) Update Rule From Memory

Write the TD(0) update rule from memory, then answer the comprehension question.

In [ ]:
# Fill in the blanks with the correct formula
td0_update = """
TD(0) value update:
  V(s) <- V(s) + alpha * [R + gamma * V(s') - V(s)]

The term [R + gamma * V(s') - V(s)] is called the ____.
This is a ___-step return (bootstrapped from V(s')).
"""

# TODO: fill in the blanks above, then answer:
q7_answer = """
Q: Is TD(0) on-policy or off-policy when used to evaluate a policy?
A: TODO

Q: What about Q-learning — is it on-policy or off-policy? Why?
A: TODO
"""

print(td0_update)
print(q7_answer)

<details><summary>▶ Answer</summary>

- The term `[R + γV(s') − V(s)]` is the **TD error** (δ).
- This is a **1-step** return.
- TD(0) for policy evaluation is **on-policy** — it evaluates the policy being followed.
- Q-learning is **off-policy** — it uses `max_a Q(s',a)` as the target, regardless of which action is actually taken by the behavior policy.
</details>

---
## Drill 2 — TD Update: Compute New Q by Hand

Given:
- State s=A, action=right, reward r=1, next state s'=B
- Q(A, right) = 0.3, Q(B, best) = 0.5
- α = 0.1, γ = 0.9

Compute the new Q(A, right) after a **Q-learning** update.

In [ ]:
# By hand first:
# TD target  = r + gamma * max Q(s', a)
#            = 1 + 0.9 * 0.5 = __
# TD error   = target - Q(s, a)
#            = __ - 0.3 = __
# New Q      = Q + alpha * TD_error
#            = 0.3 + 0.1 * __ = __

def q_learning_update(Q_sa, r, Q_s_prime_max, alpha=0.1, gamma=0.9):
    """Q-learning update. Returns new Q(s,a)."""
    td_target = r + gamma * Q_s_prime_max
    td_error  = td_target - Q_sa
    return Q_sa + alpha * td_error

new_Q = q_learning_update(Q_sa=0.3, r=1.0, Q_s_prime_max=0.5)
print(f"New Q(A, right) = {new_Q:.4f}")
print(f"Expected: 0.3 + 0.1 * (1.45 - 0.3) = 0.3 + 0.115 = 0.415")

---
## Drill 3 — SARSA vs Q-Learning: Different Targets

Given the **same transition** (s=A, a=right, r=1, s'=B),  
and the agent actually takes action `left` next (Q(B,left)=0.2, Q(B,right)=0.5),  
show that SARSA and Q-learning produce **different** updates.

In [ ]:
alpha, gamma = 0.1, 0.9
Q_Aright = 0.3
r = 1.0

# Q values at B
Q_B = {'left': 0.2, 'right': 0.5}
next_action_taken = 'left'   # SARSA uses the ACTUAL next action

# SARSA: uses Q(s', a') where a' is the ACTUAL next action chosen by policy
sarsa_target = r + gamma * Q_B[next_action_taken]
sarsa_new_Q  = Q_Aright + alpha * (sarsa_target - Q_Aright)

# Q-learning: uses max_a Q(s', a) — greedy regardless of what was done
ql_target  = r + gamma * max(Q_B.values())
ql_new_Q   = Q_Aright + alpha * (ql_target - Q_Aright)

print(f"SARSA  target: {sarsa_target:.3f}  →  new Q(A,right) = {sarsa_new_Q:.4f}")
print(f"Q-learn target: {ql_target:.3f}  →  new Q(A,right) = {ql_new_Q:.4f}")
print(f"\nDifference: {abs(sarsa_new_Q - ql_new_Q):.4f}")
print("Key insight: Q-learning is more optimistic — it assumes best future action.")

---
## Drill 4 — Monte Carlo First-Visit: Trace an Episode

For first-visit MC, we update G for the **first** time each state is visited in an episode.  
Trace through this 4-step episode and identify which updates are made.

In [ ]:
# Episode: states=[A, B, A, C], actions=[r,l,r], rewards=[0, 1, 0, 5]
# (rewards[t] received after leaving states[t])
# gamma = 0.9

def mc_first_visit(episode_states, episode_rewards, gamma=0.9):
    """First-visit MC: compute G for first occurrence of each state.
    Returns dict: state -> G computed at first visit.
    """
    G = 0.0
    visited = {}
    T = len(episode_rewards)

    # Work backwards: compute returns from end
    returns_from = {}  # returns_from[t] = G_t
    G = 0.0
    for t in range(T - 1, -1, -1):
        G = episode_rewards[t] + gamma * G
        returns_from[t] = G

    # First-visit: record G only for first occurrence of each state
    first_visit_G = {}
    seen = set()
    for t in range(T):
        s = episode_states[t]
        if s not in seen:
            first_visit_G[s] = returns_from[t]
            seen.add(s)
            print(f"  t={t}: First visit to {s}, G_{t} = {returns_from[t]:.4f}")
        else:
            print(f"  t={t}: State {s} already seen — skip (first-visit MC)")

    return first_visit_G

states  = ['A', 'B', 'A', 'C']
rewards = [0,    1,   0,   5]    # reward received when leaving each state

print("Episode:", list(zip(states, rewards)))
print("First-visit updates:")
fv_G = mc_first_visit(states, rewards)
print("\nFirst-visit returns:", {k: f'{v:.4f}' for k, v in fv_G.items()})

---
## Drill 5 — n-Step Return

The **n-step return** bootstraps after n steps:

$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \ldots + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})$$

Compute the 2-step return for a sequence.

In [ ]:
def n_step_return(rewards, V_bootstrap, gamma=0.9, n=2):
    """Compute G^(n)_0 from rewards[0..n-1] then bootstrap with V_bootstrap.
    rewards: list of rewards (length >= n)
    V_bootstrap: V(S_n), the value at the bootstrap state
    """
    G = V_bootstrap * (gamma ** n)
    for t, r in enumerate(rewards[:n]):
        G += (gamma ** t) * r
    return G

# Example: rewards from t=0 are [1, 2, 3, 4], n=2, V(S_2) = 5.0
rewards = [1, 2, 3, 4]
V_s2 = 5.0

G2 = n_step_return(rewards, V_bootstrap=V_s2, gamma=0.9, n=2)
print(f"2-step return: {G2:.4f}")
print(f"By hand: 1 + 0.9*2 + 0.9^2*5.0 = {1 + 0.9*2 + 0.81*5.0:.4f}")

print()
# Spectrum: n=1 is TD(0), n=inf is MC
for n in [1, 2, 3, len(rewards)]:
    # For large n, use actual rewards (no bootstrap needed if n >= len)
    bootstrap = V_s2 if n < len(rewards) else 0.0
    G = n_step_return(rewards, V_bootstrap=bootstrap, gamma=0.9, n=min(n, len(rewards)))
    print(f"n={n}: G = {G:.4f}")

---
## Drill 6 — Debug: SARSA Using `max` Instead of Actual Next Action

The code below claims to be SARSA, but uses `max` for the next-state value.  
That makes it Q-learning. Find the bug and fix it.

In [ ]:
# BUGGY SARSA — find the bug
def sarsa_update_buggy(Q, s, a, r, s_prime, alpha=0.1, gamma=0.9):
    """SARSA should use Q(s', a') where a' is the ACTUAL next action.
    But this implementation uses max_a Q(s', a) instead.
    """
    # BUG: this is Q-learning, not SARSA!
    best_next_Q = max(Q.get((s_prime, a_next), 0.0)
                      for a_next in range(4))
    td_error = r + gamma * best_next_Q - Q.get((s, a), 0.0)
    new_Q = Q.get((s, a), 0.0) + alpha * td_error
    return new_Q

# TODO: Write the correct SARSA update (requires knowing the actual next action a_prime)
def sarsa_update_fixed(Q, s, a, r, s_prime, a_prime, alpha=0.1, gamma=0.9):
    """Correct SARSA: uses Q(s', a') where a' was selected by the policy."""
    # TODO: fix this function
    pass

# Test
Q = {('A', 'right'): 0.3, ('B', 'left'): 0.2, ('B', 'right'): 0.5}
result = sarsa_update_fixed(Q, 'A', 'right', r=1.0, s_prime='B', a_prime='left')
print(f"Fixed SARSA Q(A,right) = {result}")
# Expected: 0.3 + 0.1*(1 + 0.9*0.2 - 0.3) = 0.3 + 0.1*(0.88) = 0.388

<details><summary>▶ Solution</summary>

```python
def sarsa_update_fixed(Q, s, a, r, s_prime, a_prime, alpha=0.1, gamma=0.9):
    # Use the ACTUAL next action a_prime, not the max
    next_Q = Q.get((s_prime, a_prime), 0.0)
    td_error = r + gamma * next_Q - Q.get((s, a), 0.0)
    return Q.get((s, a), 0.0) + alpha * td_error
```

**Why it matters:** SARSA evaluates the actual policy being followed (on-policy), while Q-learning evaluates the optimal policy regardless. In cliff-walking, this causes them to learn different safe/optimal routes.
</details>

---
## Drill 7 — When Does Q-Learning Find a Riskier Path?

Conceptual + code demonstration: SARSA vs Q-learning in the **cliff-walking** environment.

In [ ]:
# Cliff Walking: 4x12 grid. Bottom row has a cliff (except start and goal).
# SARSA (on-policy): learns to hug the wall (safe path, longer)
# Q-learning (off-policy): learns optimal path along the cliff edge
# BUT during training, Q-learning suffers more falls (due to epsilon exploration)

explanation = """
Q-learning vs SARSA on Cliff Walking:

Q-learning:
  - Learns the OPTIMAL path (shortest — along cliff edge)
  - But during training with epsilon-greedy, occasionally steps off the cliff
  - Lower asymptotic performance (converges to optimal), higher training cost

SARSA:
  - Learns a SAFE path (further from cliff)
  - Accounts for the exploration noise in its own updates
  - Higher asymptotic performance during training (fewer cliff falls)

Key insight: Q-learning is off-policy — it ignores the risk from exploration.
             SARSA is on-policy — it learns to avoid actions that might
             trigger exploration near dangerous states.
"""
print(explanation)

# TODO: In which scenario would you PREFER Q-learning over SARSA?
your_answer = "TODO: When is Q-learning the better choice?"

---
## Drill 8 — Incremental Mean Update (Review)

This is the foundation of all Q-value updates. Implement it from scratch.

In [ ]:
def incremental_mean(Q, n, new_value):
    """Q_{n+1} = Q_n + (1/n) * (new_value - Q_n)"""
    return Q + (1.0 / n) * (new_value - Q)

# Demonstrate equivalence to simple mean:
samples = [3.0, 1.0, 4.0, 1.0, 5.0, 9.0, 2.0, 6.0]

Q_incr = 0.0
for i, x in enumerate(samples, start=1):
    Q_incr = incremental_mean(Q_incr, i, x)

simple_mean = sum(samples) / len(samples)
print(f"Incremental mean: {Q_incr:.6f}")
print(f"Simple mean:      {simple_mean:.6f}")
print(f"Match: {abs(Q_incr - simple_mean) < 1e-10}")

# Connection to TD: using alpha=1/n gives the same result.
# Using constant alpha (e.g., 0.1) gives exponentially weighted average
# — more recent samples count more. Better for non-stationary problems!

---
## Drill 9 — ε-Greedy Probability Calculation

**Conceptual + math:** With ε = 0.1 and k = 4 actions, what is the probability of selecting the greedy action?

In [ ]:
def epsilon_greedy_prob(epsilon, n_actions, is_greedy=True):
    """Probability that ε-greedy selects the greedy action (or a non-greedy one)."""
    # With prob (1-epsilon): greedy action
    # With prob epsilon: uniform random over all n_actions actions
    if is_greedy:
        return (1 - epsilon) + epsilon / n_actions
    else:
        return epsilon / n_actions

epsilon, k = 0.1, 4
p_greedy = epsilon_greedy_prob(epsilon, k, is_greedy=True)
p_other  = epsilon_greedy_prob(epsilon, k, is_greedy=False)

print(f"P(greedy action)     = {p_greedy:.4f}")
print(f"P(each other action) = {p_other:.4f}")
print(f"Sum check (4 actions total): {p_greedy + 3*p_other:.6f}  (should be 1.0)")
print()
print("Manual calculation:")
print(f"  P(greedy) = (1-{epsilon}) + {epsilon}/{k} = {1-epsilon} + {epsilon/k} = {p_greedy}")

---
## Drill 10 — TD(0) One-Step Update Function

Implement a complete TD(0) update function for state-value estimation.

In [ ]:
def td0_update(V, s, r, s_prime, alpha=0.1, gamma=0.9, done=False):
    """TD(0) update for V(s).
    Returns new V(s) and the TD error.
    done: True if s_prime is terminal.
    """
    V_s_prime = 0.0 if done else V.get(s_prime, 0.0)
    td_error = r + gamma * V_s_prime - V.get(s, 0.0)
    new_V_s = V.get(s, 0.0) + alpha * td_error
    return new_V_s, td_error

# Simulate a simple random walk episode
import random
random.seed(42)

V = {'A': 0.0, 'B': 0.0, 'C': 0.0, 'D': 0.0, 'E': 0.0}
# Simulate 100 episodes of a 5-state random walk
for episode in range(500):
    s = 'C'   # start in middle
    while True:
        s_prime = random.choice(['left', 'right'])
        states_list = ['T_left', 'A', 'B', 'C', 'D', 'E', 'T_right']
        idx = states_list.index(s)
        next_s = states_list[idx + (1 if s_prime=='right' else -1)]

        done = next_s in ('T_left', 'T_right')
        r = 1.0 if next_s == 'T_right' else 0.0

        if s in V:
            new_v, _ = td0_update(V, s, r, next_s, alpha=0.1, done=done)
            V[s] = new_v
        if done:
            break
        s = next_s

print("Learned V after 500 episodes:")
for state in ['A', 'B', 'C', 'D', 'E']:
    print(f"  V({state}) = {V[state]:.4f}")
print("\nTrue values (proportional to distance from right terminal):")
true_V = {'A': 1/6, 'B': 2/6, 'C': 3/6, 'D': 4/6, 'E': 5/6}
for state in ['A', 'B', 'C', 'D', 'E']:
    print(f"  True V({state}) = {true_V[state]:.4f}")

---
## Drill 11 — Full Q-Learning Agent

Implement a complete Q-learning agent and run it on a simple 5-state chain.

In [ ]:
import random
from collections import defaultdict

class QLearningAgent:
    def __init__(self, n_actions=2, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.Q = defaultdict(float)
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        q_vals = [self.Q[(state, a)] for a in range(self.n_actions)]
        return q_vals.index(max(q_vals))

    def update(self, s, a, r, s_prime, done):
        if done:
            target = r
        else:
            max_next_Q = max(self.Q[(s_prime, a2)] for a2 in range(self.n_actions))
            target = r + self.gamma * max_next_Q
        self.Q[(s, a)] += self.alpha * (target - self.Q[(s, a)])

# Chain MDP: states 0-4, actions 0=left, 1=right
# right from 4 gives reward 1 (terminal), else 0
def chain_step(s, a):
    if a == 1:  # right
        s_prime = s + 1
    else:        # left
        s_prime = max(s - 1, 0)
    done = s_prime >= 5
    r = 1.0 if done else 0.0
    return s_prime, r, done

random.seed(0)
agent = QLearningAgent(n_actions=2)
rewards_per_ep = []

for ep in range(500):
    s = 0
    total_r = 0
    for _ in range(20):  # max 20 steps
        a = agent.select_action(s)
        s_prime, r, done = chain_step(s, a)
        agent.update(s, a, r, s_prime, done)
        total_r += r
        s = s_prime
        if done:
            break
    rewards_per_ep.append(total_r)

print("Q-table (state, action -> Q):")
for s in range(5):
    for a in range(2):
        print(f"  Q({s},{['left','right'][a]}) = {agent.Q[(s,a)]:.4f}")

recent_avg = sum(rewards_per_ep[-100:]) / 100
print(f"\nAvg reward (last 100 eps): {recent_avg:.3f}  (should be ~1.0)")

---
## Drill 12 — Full SARSA Agent

Adapt the Q-learning agent to use SARSA updates.

In [ ]:
class SARSAAgent:
    def __init__(self, n_actions=2, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.Q = defaultdict(float)
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        q_vals = [self.Q[(state, a)] for a in range(self.n_actions)]
        return q_vals.index(max(q_vals))

    def update(self, s, a, r, s_prime, a_prime, done):
        """SARSA: uses the ACTUAL next action a_prime."""
        if done:
            target = r
        else:
            # KEY DIFFERENCE: use Q(s', a') not max Q(s', *)
            target = r + self.gamma * self.Q[(s_prime, a_prime)]
        self.Q[(s, a)] += self.alpha * (target - self.Q[(s, a)])

random.seed(0)
sarsa_agent = SARSAAgent(n_actions=2)

for ep in range(500):
    s = 0
    a = sarsa_agent.select_action(s)
    for _ in range(20):
        s_prime, r, done = chain_step(s, a)
        a_prime = sarsa_agent.select_action(s_prime) if not done else 0
        sarsa_agent.update(s, a, r, s_prime, a_prime, done)
        s, a = s_prime, a_prime
        if done:
            break

print("SARSA Q-table:")
for s in range(5):
    for a_idx in range(2):
        print(f"  Q({s},{['left','right'][a_idx]}) = {sarsa_agent.Q[(s,a_idx)]:.4f}")

---
## Drill 13 — Double Q-Learning Motivation

**Why does regular Q-learning overestimate?** Demonstrate the maximization bias.

In [ ]:
import random

# Maximization bias demo:
# True Q values are all 0, but with noise we'll see overestimation.
random.seed(0)

n_actions = 10
n_samples = 5   # samples per action
true_Q = 0.0    # all actions have Q=0

estimates = []
for trial in range(1000):
    # Sample noisy estimates for each action
    noisy_Q = [random.gauss(true_Q, 1.0) for _ in range(n_actions)]
    # Q-learning would use: max of these estimates
    estimates.append(max(noisy_Q))

avg_max = sum(estimates) / len(estimates)
print(f"True max Q = {true_Q:.1f}")
print(f"Avg of max(noisy_Q): {avg_max:.4f}")
print(f"Overestimation: {avg_max - true_Q:.4f}")
print()
print("This is maximization bias — using the same samples to SELECT and EVALUATE")
print("the max action leads to systematic overestimation.")
print("Double Q-learning fixes this by using separate Q1, Q2.")

---
## Drill 14 — TD vs MC Bias-Variance

**Conceptual:** Fill in the bias-variance tradeoff table and verify with a small experiment.

In [ ]:
bias_variance_table = """
Method          | Bias          | Variance   | Data Efficiency
----------------|---------------|------------|----------------
TD(0)           | Higher (boot) | Lower      | High
MC              | Zero          | Higher     | Low
n-step (small n)| Medium        | Medium     | Medium
n-step (large n)| Lower         | Higher     | Lower

Key insight:
- MC uses the actual return → unbiased but noisy (many random steps)
- TD bootstraps from V(s') → introduces bias but reduces variance
- The TD bias comes from incorrect V(s') estimates (which improve over time)
"""
print(bias_variance_table)

# TODO: Why does TD(0) still converge despite the bias?
your_answer = "TODO: explain why TD converges despite bootstrapping bias"

---
## Drill 15 — Put It All Together: Tabular RL Algorithms Summary

Match each algorithm to its update target and on/off-policy status.

In [ ]:
# Complete this table by filling in the TODO fields
algorithms = [
    {
        "name": "Monte Carlo",
        "target": "G_t (actual return from episode)",
        "on_off_policy": "On-policy (can be adapted to off-policy with importance sampling)",
        "bootstraps": False,
        "needs_full_episode": True,
    },
    {
        "name": "TD(0) / SARSA",
        "target": "R + gamma * Q(s', a')  where a' ~ pi",
        "on_off_policy": "On-policy",
        "bootstraps": True,
        "needs_full_episode": False,
    },
    {
        "name": "Q-Learning",
        "target": "R + gamma * max_a Q(s', a)",
        "on_off_policy": "Off-policy",
        "bootstraps": True,
        "needs_full_episode": False,
    },
    {
        "name": "Expected SARSA",
        "target": "TODO: what is the expected SARSA target?",
        "on_off_policy": "TODO: on or off policy?",
        "bootstraps": True,
        "needs_full_episode": False,
    },
]

print(f"{'Algorithm':<20} {'On/Off':<12} {'Bootstraps':<12} {'Target'}")
print("-" * 80)
for alg in algorithms:
    print(f"{alg['name']:<20} {alg['on_off_policy'][:11]:<12} {str(alg['bootstraps']):<12} {alg['target']}")

<details><summary>▶ Expected SARSA Answer</summary>

**Expected SARSA target:**
$$R + \gamma \sum_{a'} \pi(a'|s') Q(s', a')$$

It uses the **expected** value under the policy instead of the actual next action (SARSA) or the max (Q-learning). This reduces variance compared to SARSA.

**On/off-policy:** It can be either! If π is the same as the behavior policy → on-policy. If π is the greedy policy (while behaving with epsilon-greedy) → off-policy. When π = greedy, Expected SARSA == Q-learning.
</details>

---
## Summary

| Drill | Topic |
|-------|-------|
| 1 | TD(0) update rule, on/off-policy |
| 2 | Q-learning update by hand |
| 3 | SARSA vs Q-learning targets |
| 4 | MC first-visit trace |
| 5 | n-step return |
| 6 | Debug SARSA |
| 7 | Q-learning riskier path |
| 8 | Incremental mean |
| 9 | ε-greedy probability |
| 10 | TD(0) full implementation |
| 11 | Q-learning agent |
| 12 | SARSA agent |
| 13 | Maximization bias |
| 14 | Bias-variance tradeoff |
| 15 | Algorithm comparison table |

**Next:** Volume 3 — Function Approximation.